In [ ]:
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from yandex_cloud_ml_sdk import YCloudML


FAISS_INDEX_PATH = "faiss.index"
METADATA_PATH = "metadata.json"

# 🧠 Загружаем модель для эмбеддингов
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 📂 Загружаем FAISS индекс
index = faiss.read_index(FAISS_INDEX_PATH)

# 📄 Загружаем метаданные
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)


def search(query, model, index, metadata, k=20):
    # Генерация вектора для запроса
    q_vec = model.encode([query], convert_to_numpy=True)
    D, I = index.search(q_vec, k)
    results = []
    for idx, dist in zip(I[0], D[0]):
        if idx == -1:
            continue
        result = metadata[idx].copy()
        result["score"] = float(dist)
        results.append(result)
    return results


user_query = input("Введите ваш запрос: ")

results = search(user_query, model, index, metadata, k=20)

# print(f"\n Запрос пользователя: {user_query}")

# for i, r in enumerate(results, 1):
#     print(f"\n=== Результат {i} ===")
#     print(f"Семантическая близость: {r['score']:.4f}")
#     print(f"Текст:\n{r['chunk'][:500]}...")
    
    
def format_results(results, max_excerpt_length: int = 350) -> str:
    lines = []
    for i, r in enumerate(results, 1):
        title = r.get("title", "Без названия").strip()
        chunk = r.get("chunk", "").replace("\n", " ").strip()
        
        # Обрезаем отрывок, если он слишком длинный
        if len(chunk) > max_excerpt_length:
            chunk = chunk[:max_excerpt_length].rstrip() + "..."
        
        line = f"[{i}] {title} — {chunk}"
        lines.append(line)
    
    return "\n".join(lines)

documents = format_results(results)

messages_to_llm = [
    {
        "role": "system",
        "text": f"""
### Роль
Ты — крупная русскоязычная LLM‑модель‑ассистент.  
Твоя задача — аккуратно ответить на вопрос пользователя, используя ТОЛЬКО информацию из предоставленного списка документов.  
Если в документах нет нужной информации, честно скажи «Не нашёл подтверждений».  
Избегай домыслов и галлюцинаций.
Ты помощник, который сначала размышляет, а потом отвечает. Всегда пиши свои шаги.

### Шаги работы
0. Опиши шаги которые ты будешь делать перед тем как дать ответ.  
1. Внимательно прочитай все документы из блока <Документы>.  
2. Определи, какие из них действительно релевантны вопросу.  
3. Сконспектируй ключевые факты (можешь делать пометки для себя, но не показывай их пользователю).  
4. Сформулируй итоговый ответ на русском, опираясь только на подтверждённые факты.  
5. В конце ответа проставь цитаты вида [1], [2] — это номера документов из блока <Документы>, которые подтвердили конкретное утверждение.

### Формат выдачи
Ответ должен состоять из трех частей:
**Порядок размышления:** Шаги предпринятые перед ответом.
**A. Краткий ответ** (1‑3 предложения).  
**B. Развёрнутое объяснение** (по пунктам), где каждый тезис снабжён ссылкой‑номером на источник в квадратных скобках.

### <Документы>
{documents}

### <Твой ответ>
(Соблюдай формат Порядок размышления. A. и B., как описано выше) 
        """,
    },
    {
        "role": "user",
        "text": user_query,
    },
]

sdk = YCloudML(
        folder_id="",
        auth="",
    )

model = sdk.models.completions("yandexgpt")
operation = model.run_deferred(messages_to_llm)

result = operation.wait()
print(f"Сообщение пользователя: {user_query}")
print(result.alternatives[0].text)

Сообщение пользователя: Ты видел что-то про swordfish в документации?
Порядок размышления:
1. Прочитал все документы из блока <Документы>.
2. Определил, что только документ [1] содержит упоминание слова «swordfish».
3. Сформулировал ответ на основе найденной информации.

A. Да, в документации есть упоминание слова «swordfish».

B. В одном из документов содержится следующая информация: «Суперпароль root: swordfish» [1].
